# 해석적 Denoiser 실험
`nu=3` 은 학습 탓인가, 샘플러 탓인가

## 0. 셋업

In [23]:
%load_ext autoreload
%autoreload 2

import os, sys, time

sys.path.insert(0, os.path.abspath("."))

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats

import t_diffusion as td

import matplotlib.font_manager as fm
_have = {f.name for f in fm.fontManager.ttflist}
KFONT = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans KR"]
              if f in _have), None)
if KFONT:
    plt.rcParams["font.family"] = KFONT
    plt.rcParams["mathtext.fontset"] = "custom"
    for _k, _v in [("rm", "DejaVu Sans"), ("it", "DejaVu Sans:italic"),
                   ("bf", "DejaVu Sans:bold"), ("sf", "DejaVu Sans"),
                   ("cal", "DejaVu Sans:italic"), ("tt", "DejaVu Sans Mono")]:
        plt.rcParams[f"mathtext.{_k}"] = _v
plt.rcParams["axes.unicode_minus"] = False

NU_DATA = 3.0                          # 데이터 분포의 자유도
NU_MODEL = [3.0, 10.0, 30.0, 500.0]    # 확산 잡음의 자유도
LOC, SCALE = 0.0, 1.0
N_TOTAL, N_TRAIN = 50000, 10000
T = 1000
STEPS, BATCH, LR, HIDDEN = 20000, 128, 1e-3, 128
N_NET = 20000          # 학습된 denoiser 로 뽑을 표본 수
N_EXACT = 8000         # 해석적 denoiser 는 매 스텝 (n x M) 행렬을 만든다
M_DATA = 10000         # denoiser 가 참조하는 표본 수 (= 경험적 사전분포)
CHUNK = 4000
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device =", DEVICE, "| steps =", STEPS, "| nu =", NU_MODEL)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device = cuda | steps = 20000 | nu = [3.0, 10.0, 30.0, 500.0]


## 1. 데이터

In [24]:
def make_data():
    rng = np.random.default_rng(SEED)
    pool = stats.t.rvs(df=NU_DATA, loc=LOC, scale=SCALE, size=N_TOTAL, random_state=rng)
    idx = rng.permutation(N_TOTAL)
    return pool[idx[:N_TRAIN]], pool[idx[N_TRAIN:]]

train_np, held = make_data()
x0 = torch.tensor(train_np, dtype=torch.float32).unsqueeze(-1)

X0 = torch.tensor(train_np[:M_DATA], dtype=torch.float32, device=DEVICE)

sch = td.Schedule(T, device=DEVICE)

print(f"data ~ t(nu={NU_DATA:g}) | train {len(train_np)} | held-out {len(held)}")
print(f"denoiser 참조 표본 M = {X0.numel()}  (min {X0.min():.2f}, max {X0.max():.2f})")

data ~ t(nu=3) | train 10000 | held-out 40000
denoiser 참조 표본 M = 10000  (min -15.43, max 43.62)


## 2. 해석적 denoiser

In [25]:
@torch.no_grad()
def exact_denoiser(x, i, nu):
    out = torch.empty_like(x)
    for s in range(0, x.shape[0], CHUNK):
        z = (x[s:s + CHUNK] - sch.mu[i] * X0[None, :]) / sch.sigma[i]     # (b, M)
        w = torch.softmax(-0.5 * (nu + 1.0) * torch.log1p(z * z / nu), dim=1)
        out[s:s + CHUNK] = (w @ X0)[:, None]                               # E[x_0 | x_t]
    return out

## 3. 공통 샘플러

In [27]:
@torch.no_grad()
def sample_chain(nu, denoiser="net", model=None, n=None, seed=SEED):
    n = (N_EXACT if denoiser == "exact" else N_NET) if n is None else n
    torch.manual_seed(seed)
    d = 1

    x = td.sample_mvt_noise(n, d, nu, DEVICE) * sch.sigma[T - 1]

    for i in reversed(range(T)):
        if denoiser == "net":
            D = model(x, torch.full((n,), i, device=DEVICE, dtype=torch.long))
        else:
            D = exact_denoiser(x, i, nu)

        mu_theta = sch.ct[i] * x + sch.c0[i] * D
        if i == 0:
            x = mu_theta
            break

        delta = ((x - sch.mu[i] * D) / sch.sigma[i]).pow(2).sum(-1, keepdim=True)
        s = (nu + delta) / (nu + d)
        x = mu_theta + (sch.sigmabar2[i] * s).sqrt() * td.sample_mvt_noise(n, d, nu + d, DEVICE)

    return x.squeeze(-1).cpu().numpy().astype(np.float64)

## 4. 평가 지표

In [28]:
def evaluate(g):
    g = np.asarray(g, dtype=np.float64)
    g = g[np.isfinite(g)]
    df_hat, loc_hat, sc_hat = stats.t.fit(g)
    a = np.sort(np.abs(g))[::-1]
    k = max(10, int(0.1 * len(a)))
    a = a[: k + 1]
    return {
        "ks": float(stats.kstest(g, "t", args=(NU_DATA, LOC, SCALE)).statistic),
        "w1": float(stats.wasserstein_distance(g, held)),
        "mle_df": float(df_hat),
        "mle_scale": float(sc_hat),
        "hill": float(1.0 / np.mean(np.log(a[:k] / a[k]))),
        "q999": float(np.quantile(g, 0.999)),
        "iqr": float(np.subtract(*np.percentile(g, [75, 25]))),
        "p5": float(np.mean(np.abs(g) > 5)),
    }

TRUE = {"mle_df": NU_DATA, "mle_scale": SCALE, "hill": NU_DATA,
        "q999": float(stats.t.ppf(0.999, NU_DATA, LOC, SCALE)),
        "iqr": float(2 * stats.t.ppf(0.75, NU_DATA, LOC, SCALE)),
        "p5": float(2 * stats.t.sf(5, NU_DATA, LOC, SCALE))}

HDR = (f"{'nu':>5} {'denoiser':>9} | {'KS':>7} {'W1':>7} "
       f"{'MLEdof':>7} {'scale':>6} {'Hill':>5} {'q99.9':>6} {'IQR':>6} {'P|x|>5':>7}")

def row(nu, dn, m):
    return (f"{nu:5g} {dn:>9} | {m['ks']:7.4f} {m['w1']:7.4f} "
            f"{m['mle_df']:7.2f} {m['mle_scale']:6.3f} {m['hill']:5.2f} "
            f"{m['q999']:6.2f} {m['iqr']:6.3f} {m['p5']:7.4f}")

def true_row():
    return (f"{'TRUE':>5} {'':>9} | {'-':>7} {'-':>7} "
            f"{TRUE['mle_df']:7.2f} {TRUE['mle_scale']:6.3f} {TRUE['hill']:5.2f} "
            f"{TRUE['q999']:6.2f} {TRUE['iqr']:6.3f} {TRUE['p5']:7.4f}")

## 5. 학습 (비교군)

In [29]:
models = {}
t_all = time.time()
for nu in NU_MODEL:
    t0 = time.time()
    print(f"  nu={nu:<6g} training ... ", end="", flush=True)
    m, _ = td.train(x0, nu, T=T, steps=STEPS, batch=BATCH, lr=LR,
                    hidden=HIDDEN, device=DEVICE, seed=SEED)
    models[nu] = m
    print(f"{time.time() - t0:.1f}s")
print(f"done in {time.time() - t_all:.1f}s")

  nu=3      training ... 35.6s
  nu=10     training ... 35.9s
  nu=30     training ... 30.1s
  nu=500    training ... 29.4s
done in 131.0s


## 6. 본 실험 — 학습된 denoiser vs 해석적 denoiser

In [ ]:
res = {}
print(HDR)
print("-" * len(HDR))
print(true_row())
print("-" * len(HDR))

for nu in NU_MODEL:
    for dn in ["net", "exact"]:
        g = sample_chain(nu, dn, models[nu] if dn == "net" else None)
        res[(nu, dn)] = evaluate(g)
        print(row(nu, dn, res[(nu, dn)]), flush=True)
    print("-" * len(HDR))

## 7. 분포 그림

In [ ]:
BINS, LIM = 120, 12.0
grid = np.linspace(-LIM, LIM, 2000)

gens = {}
for nu in NU_MODEL:
    gens[(nu, "net")] = sample_chain(nu, "net", models[nu])
    gens[(nu, "exact")] = sample_chain(nu, "exact")

fig, axes = plt.subplots(2, len(NU_MODEL), figsize=(3.2 * len(NU_MODEL), 6.0),
                         sharey=True, sharex=True)
axes = np.atleast_2d(axes)
for r_, dn in enumerate(["net", "exact"]):
    for c_, nu in enumerate(NU_MODEL):
        ax = axes[r_, c_]
        g = gens[(nu, dn)]
        floor = 1.0 / (len(g) * (2 * LIM / BINS)) / 2
        ax.hist(np.clip(g, -LIM, LIM), bins=BINS, range=(-LIM, LIM),
                density=True, alpha=0.55, label="generated")
        ax.plot(grid, stats.t.pdf(grid, NU_DATA, LOC, SCALE), "k-", lw=1.2,
                label=f"true t({NU_DATA:g})")
        ax.set_yscale("log")
        ax.set_ylim(floor, 1)
        ax.set_title(f"nu={nu:g}  ({dn})", fontsize=9)
        ax.tick_params(labelsize=7)
axes[0, 0].legend(fontsize=7)
fig.suptitle("density (log y) — 위: 학습된 denoiser / 아래: 해석적 denoiser", fontsize=11)
fig.tight_layout()
plt.show()